In [1]:
#This will be for all my imports so it is easy to check my environment: 
import numpy as np
import pandas as pd
import xarray as xr

In [2]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

In [3]:
import sys
import torch
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
import torchvision.transforms as transforms

In [4]:
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [5]:
import datetime
from datetime import timedelta

In [6]:
import cftime
import torchvision.transforms as T
import dask

In [7]:
print("CUDA Version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

CUDA Version: 12.4
CUDA available: False


In [8]:
#This was created by Maria to help import the correct Datetime for my variables from GEOS 
def fixthetime(ds):
    newdates = [
        pd.to_datetime(
            ds.Datetime.attrs['units'][-19:]) + timedelta(hours=int(i)) for i in ds.Datetime.values
    ]
    ds['Datetime'] = newdates
    return ds

In [9]:
#This is to fix the time on the LWI variable. 
def fixthetime_LWI(ds):
    # Extract the base time from the 'units' attribute
    base_time_str = ds.Datetime.attrs['units'].split('since')[-1].strip()
    base_time = pd.to_datetime(base_time_str)

    # Get the minute offsets
    minute_offsets = ds.Datetime.values.astype(float)

    # Create real timestamps
    new_times = base_time + pd.to_timedelta(minute_offsets, unit='m')

    # Replace the time coordinate
    ds = ds.assign_coords(Datetime=('Datetime', new_times))
    return ds

In [10]:
#Bring in the USA Data. For the FEATURES, I am bringing in the raw data. It has been cleaned, but there have been no transformations or 
#standardizations on it. 
cape_usa = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/USA_new/cape_usa_new.nc',
                                   decode_times=False, preprocess=fixthetime)
cldfrac_usa = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/USA_new/cloudfrac_usa_new.nc',
                                   decode_times=False, preprocess=fixthetime)
cldht_usa = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/USA_new/cldht_usa_new.nc',
                                   decode_times=False, preprocess=fixthetime)
crh_usa = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/USA_new/crh_usa_new.nc',
                                   decode_times=False, preprocess=fixthetime)
iceflux_usa = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/USA_new/iceflux_usa_new.nc',
                                   decode_times=False, preprocess=fixthetime)
iwc440_usa = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/USA_new/iwc440_usa_new.nc',
                                   decode_times=False, preprocess=fixthetime)
lcl_usa = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/USA_new/LCL_usa_new.nc',
                                   decode_times=False, preprocess=fixthetime)
massflux_usa = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/USA_new/massflux_usa_new.nc',
                                   decode_times=False, preprocess=fixthetime)
maxcloudfrac_usa = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/USA_new/maxfrac_usa_new.nc',
                                   decode_times=False, preprocess=fixthetime)
mseratio_usa = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/USA_new/mseratio_usa_new.nc',
                                   decode_times=False, preprocess=fixthetime)
precon_usa = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/USA_new/convpre_usa_new.nc',
                                   decode_times=False, preprocess=fixthetime)
pretot_usa = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/USA_new/totprecip_usa_new.nc',
                                   decode_times=False, preprocess=fixthetime)
tlapse_usa = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/USA_new/tlapse_usa_new.nc',
                               decode_times=False, preprocess=fixthetime)
LWI_usa = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/USA_new/LWI_usa_new.nc',
                                decode_times=False, preprocess=fixthetime)

In [12]:
#Bring in the Amazon Data. For the FEATURES, I am bringing in the raw data. It has been cleaned, but there have been no transformations or 
#standardizations on it. 
cape_amazon = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/Amazon_new/cape_amazon_new.nc',
                                   decode_times=False, preprocess=fixthetime)
cldfrac_amazon = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/Amazon_new/cloudfrac_amazon_new.nc',
                                   decode_times=False, preprocess=fixthetime)
cldht_amazon = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/Amazon_new/cldht_amazon_new.nc',
                                   decode_times=False, preprocess=fixthetime)
crh_amazon = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/Amazon_new/crh_amazon_new.nc',
                                   decode_times=False, preprocess=fixthetime)
iceflux_amazon = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/Amazon_new/iceflux_amazon_new.nc',
                                   decode_times=False, preprocess=fixthetime)
iwc440_amazon = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/Amazon_new/iwc440_amazon_new.nc',
                                   decode_times=False, preprocess=fixthetime)
lcl_amazon = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/Amazon_new/LCL_amazon_new.nc',
                                   decode_times=False, preprocess=fixthetime)
massflux_amazon = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/Amazon_new/massflux_amazon_new.nc',
                                   decode_times=False, preprocess=fixthetime)
maxcloudfrac_amazon = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/Amazon_new/maxfrac_amazon_new.nc',
                                   decode_times=False, preprocess=fixthetime)
mseratio_amazon = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/Amazon_new/mseratio_amazon_new.nc',
                                   decode_times=False, preprocess=fixthetime)
precon_amazon = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/Amazon_new/convpre_amazon_new.nc',
                                   decode_times=False, preprocess=fixthetime)
pretot_amazon = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/Amazon_new/totprecip_amazon_new.nc',
                                   decode_times=False, preprocess=fixthetime)
tlapse_amazon = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/Amazon_new/tlapse_amazon_new.nc',
                                   decode_times=False, preprocess=fixthetime)
LWI_data_amazon = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/Amazon_new/LWI_amazon_new.nc',
                                    decode_times=False,preprocess=fixthetime)

In [17]:
Lopez_usa = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/USA_new/lopez_usa.nc',
                                   decode_times=False, preprocess=fixthetime)
Lopez_amazon = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Features/Amazon_new/lopez_amazon.nc',
                                   decode_times=False, preprocess=fixthetime)

In [18]:
#Combine all the features into one dataset for ease. USA
usa_features = xr.Dataset(
    {
    "cape": cape_usa['cape'], 
    "cldfrac": cldfrac_usa['cldfrac_conv_440'],
    "cldht": cldht_usa['z_cldht'],
    "crh": crh_usa['crh'],
    "iceflux": iceflux_usa['pfi_cn_colmax'],
    "iwc440": iwc440_usa['iwc_440'],
    "lcl": lcl_usa['lcl_moist'],
    "massflux": massflux_usa['cnv_mfc_440'],
    "maxcloudfrac": maxcloudfrac_usa['clcn_colmax'],
    "mseratio": mseratio_usa['mse_ratio'],
    "precon": precon_usa['precon'],
    "pretot": pretot_usa['pretot'],
    "tlapse": tlapse_usa['t_lapserate']
    }
)

In [15]:
usa_aerosols = xr.Dataset(
    {
        "pmcoarse": pmcoarse_usa['pm_coarse'],
        "pmfine": pmfine_usa['pm_fine'],
        "bc": BC_usa['tau_bc_ext'],
        "br": BR_usa['tau_br_ext'],
        "oc": OC_usa['tau_oc_ext'],
        "ss": seasalt_usa['tau_ss_ext'],
        "su": sulfate_usa['tau_su_ext'],
        "nitrate": nitrate_usa['tau_ni_ext'],
        "dust": dust_usa['tau_du_ext']
    }
)

In [19]:
#Combine all the features into one dataset for ease. Amazon
amazon_features = xr.Dataset(
    {
    "cape": cape_amazon['cape'], 
    "cldfrac": cldfrac_amazon['cldfrac_conv_440'],
    "cldht": cldht_amazon['z_cldht'],
    "crh": crh_amazon['crh'],
    "iceflux": iceflux_amazon['pfi_cn_colmax'],
    "iwc440": iwc440_amazon['iwc_440'],
    "lcl": lcl_amazon['lcl_moist'],
    "massflux": massflux_amazon['cnv_mfc_440'],
    "maxcloudfrac": maxcloudfrac_amazon['clcn_colmax'],
    "mseratio": mseratio_amazon['mse_ratio'],
    "precon": precon_amazon['precon'],
    "pretot": pretot_amazon['pretot'],
    "tlapse": tlapse_amazon['t_lapserate']
    }
)

In [16]:
amazon_aerosols = xr.Dataset(
    {
        "pmcoarse": pmcoarse_amazon['pm_coarse'],
        "pmfine": pmfine_amazon['pm_fine'],
        "bc": BC_amazon['tau_bc_ext'],
        "br": BR_amazon['tau_br_ext'],
        "oc": OC_amazon['tau_oc_ext'],
        "ss": seasalt_amazon['tau_ss_ext'],
        "su": sulfate_amazon['tau_su_ext'],
        "nitrate": nitrate_amazon['tau_ni_ext'],
        "dust": dust_amazon['tau_du_ext']
    }
)

In [20]:
#For LWI 
lwi_usa = xr.Dataset(
    {
    "lwi": LWI_usa['lwi']
    }
)

lwi_amazon = xr.Dataset(
    {
    "lwi": LWI_data_amazon['lwi']
    }
)

In [23]:
#Now, I want to bring in the LABELS. This should be four datasets. One for the USA region and the other for the 
#Amazon region for the (x+1) and (x+e) method.. These flashes have already been logged.
flashesX1_usa = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Flash_Data/usaflashes_new.nc',
                                   decode_times=False, preprocess=fixthetime)
flashesX1_amazon = xr.open_mfdataset('/home/mseiler1/scratch.pickerin-prj/Data_Folder/Flash_Data/amazonflashes_new.nc',
                                   decode_times=False, preprocess=fixthetime)

In [24]:
Lopez_usa = Lopez_usa['Lopez_flashes_usa']
Lopez_amazon = Lopez_amazon['Lopez_flashes_usa']

In [25]:
Lopez_amazon = Lopez_amazon.rename("Lopez_flashes_amazon")

In [26]:
Lopez_usa = Lopez_usa.transpose( "Datetime", "Latitudes", "Longitudes",)  # Swap lat and lon to have it match the other variables
Lopez_amazon = Lopez_amazon.transpose( "Datetime", "Latitudes", "Longitudes",)# Swap lat and lon to have it match the other variables

In [27]:
# Suppose your Dataset has one variable, e.g., "flashes"
usa_flashesX1 = flashesX1_usa["flashes_log"]
# Add a singleton 'features' dimension
usa_flashesX1 = usa_flashesX1.expand_dims(dim={"features": [0]})
# Reorder dimensions
usa_flashesX1 = usa_flashesX1.transpose("Datetime", "features", "Latitudes", "Longitudes")
#Now the flashes are xarray dataarrays. 

In [28]:
# Suppose your Dataset has one variable, e.g., "flashes"
amazon_flashesX1 = flashesX1_amazon["flashes_log"]
# Add a singleton 'features' dimension
amazon_flashesX1 = amazon_flashesX1.expand_dims(dim={"features": [0]})
# Reorder dimensions
amazon_flashesX1 = amazon_flashesX1.transpose("Datetime", "features", "Latitudes", "Longitudes")
#Now the flashes are xarray dataarrays. 

In [26]:
#Now all the data is in and in proper format.
#The LABELS are in Dataarray Format. This code is using (x+1) logged data for the labels. 
#The FEATURES are in a Dataset Format that consists of 13 Dataarrays. 

In [29]:
#Split the datasets into 14 months and 4 months. 
#Sort time. They will be split. The 14 months for the USA and the Amazon will be combined for training. 
#The remaining 4 months will remain seperate and will be used for testing. 
usa_features1 = usa_features.sortby("Datetime") #features #This is still a Dataset
amazon_features1 = amazon_features.sortby("Datetime") #features #This is still a Dataset

In [30]:
#Splitting
usa_flashes1 = usa_flashesX1.sortby("Datetime") #labels #This is a Dataarray
amazon_flashes1 = amazon_flashesX1.sortby("Datetime") #labels #This is a Dataarray

In [31]:
#Splitting the LWI
lwi_usa1 = lwi_usa.sortby("Datetime")
lwi_amazon1 = lwi_amazon.sortby("Datetime")

In [36]:
Lopez_usa1 = Lopez_usa.sortby("Datetime")
Lopez_amazon1 = Lopez_amazon.sortby("Datetime")

In [17]:
aerosols_usa1 = usa_aerosols.sortby("Datetime")
aerosols_amazon1 = amazon_aerosols.sortby("Datetime")

In [18]:
#This will split by the month
def split_by_months(data, months_to_test=7):
    end_time = data.Datetime.values[-1]
    end = pd.to_datetime(end_time)
    start_of_test = end - pd.DateOffset(months=months_to_test)
    
    training = data.sel(Datetime=slice(None, start_of_test - pd.Timedelta(hours=1)))
    testing = data.sel(Datetime=slice(start_of_test, None))
    return training, testing

In [21]:
# Split both datasets
us_features_train, us_features_test = split_by_months(usa_features) #feautres
amazon_features_train, amazon_features_test = split_by_months(amazon_features1) #amazon features 
#I want to do the same thing for the labels as well. 
usa_flash_train, usa_flash_test = split_by_months(usa_flashes1) #labels 
amazon_flash_train, amazon_flash_test = split_by_months(amazon_flashes1) #amazon labels
#and for the LWI as well 
usa_LWI_train, usa_LWI_test = split_by_months(lwi_usa1) #labels 
amazon_LWI_train, amazon_LWI_test = split_by_months(lwi_amazon1) #amazon labels
#And for the Lopez
usa_Lopez_train, usa_Lopez_test = split_by_months(Lopez_usa1) 
amazon_Lopez_train, amazon_Lopez_test = split_by_months(Lopez_amazon1)  

In [32]:
#This will split by the month. Change each time depending on how many months need to be split. 
def split_by_months_again(data, months_to_test=1):
    end_time = data.Datetime.values[-1]
    end = pd.to_datetime(end_time)
    start_of_test = end - pd.DateOffset(months=months_to_test)
    
    training = data.sel(Datetime=slice(None, start_of_test - pd.Timedelta(hours=1)))
    testing = data.sel(Datetime=slice(start_of_test, None))
    return training, testing

In [25]:
usa_sample1_features, interim_us_features =  split_by_months_again(us_features_test)
amazon_sample1_features, interim_amazon_features = split_by_months_again(amazon_features_test)
#flashes
usa_sample1_flashes, interim_us_flashes = split_by_months_again(usa_flash_test)
amazon_sample1_flashes, interim_amazon_flashes = split_by_months_again(amazon_flash_test)
#LWI
usa_sample1_LWI, interim_us_LWI = split_by_months_again(usa_LWI_test)
amazon_sample1_LWI, interim_amazon_LWI = split_by_months_again(amazon_LWI_test)
#Lopez
usa_sample1_Lopez, interim_us_Lopez = split_by_months_again(usa_Lopez_test)
amazon_sample1_Lopez, interim_amazon_Lopez = split_by_months_again(amazon_Lopez_test)

In [29]:
usa_sample2_features, interim_us_features2 =  split_by_months_again(interim_us_features)
amazon_sample2_features, interim_amazon_features2 = split_by_months_again(interim_amazon_features)
#flashes
usa_sample2_flashes, interim_us_flashes2 = split_by_months_again(interim_us_flashes)
amazon_sample2_flashes, interim_amazon_flashes2 = split_by_months_again(interim_amazon_flashes)
#LWI
usa_sample2_LWI, interim_us_LWI2 = split_by_months_again(interim_us_LWI)
amazon_sample2_LWI, interim_amazon_LWI2 = split_by_months_again(interim_amazon_LWI)
#Lopez
usa_sample2_Lopez, interim_us_Lopez2 = split_by_months_again(interim_us_Lopez)
amazon_sample2_Lopez, interim_amazon_Lopez2 = split_by_months_again(interim_amazon_Lopez)


In [33]:
usa_sample3_features, dec_us_feature =  split_by_months_again(interim_us_features2)
amazon_sample3_features, dec_amazon_feature = split_by_months_again(interim_amazon_features2)
#flashes
usa_sample3_flashes, dec_us_flash = split_by_months_again(interim_us_flashes2)
amazon_sample3_flashes, dec_amazon_flash = split_by_months_again(interim_amazon_flashes2)
#LWI
usa_sample3_LWI, dec_us_LWI = split_by_months_again(interim_us_LWI2)
amazon_sample3_LWI, dec_amazon_LWI = split_by_months_again(interim_amazon_LWI2)
#Lopez
usa_sample3_Lopez, dec_us_Lopez = split_by_months_again(interim_us_Lopez2)
amazon_sample3_Lopez, dec_amazon_Lopez = split_by_months_again(interim_amazon_Lopez2)

In [35]:
#usa_sample3_aerosols

In [37]:
# Create fake lat/lon (since shape is 64x64).
dummy_lat = np.arange(64)
dummy_lon = np.arange(64)

us_features_train.coords["Latitudes"] = dummy_lat
us_features_train.coords["Longitudes"] = dummy_lon
usa_sample1_features.coords["Latitudes"] = dummy_lat
usa_sample1_features.coords["Longitudes"] = dummy_lon
usa_sample2_features.coords["Latitudes"] = dummy_lat
usa_sample2_features.coords["Longitudes"] = dummy_lon
usa_sample3_features.coords["Latitudes"] = dummy_lat
usa_sample3_features.coords["Longitudes"] = dummy_lon

usa_flash_train.coords["Latitudes"]= dummy_lat
usa_flash_train.coords["Longitudes"] = dummy_lon
usa_sample1_flashes.coords["Latitudes"]= dummy_lat
usa_sample1_flashes.coords["Longitudes"] = dummy_lon
usa_sample2_flashes.coords["Latitudes"]= dummy_lat
usa_sample2_flashes.coords["Longitudes"] = dummy_lon
usa_sample3_flashes.coords["Latitudes"]= dummy_lat
usa_sample3_flashes.coords["Longitudes"] = dummy_lon

amazon_features_train.coords["Latitudes"] = dummy_lat
amazon_features_train.coords["Longitudes"] = dummy_lon
amazon_sample1_features.coords["Latitudes"] = dummy_lat
amazon_sample1_features.coords["Longitudes"] = dummy_lon
amazon_sample2_features.coords["Latitudes"] = dummy_lat
amazon_sample2_features.coords["Longitudes"] = dummy_lon
amazon_sample3_features.coords["Latitudes"] = dummy_lat
amazon_sample3_features.coords["Longitudes"] = dummy_lon

amazon_flash_train.coords["Latitudes"] = dummy_lat
amazon_flash_train.coords["Longitudes"] = dummy_lon
amazon_sample1_flashes.coords["Latitudes"] = dummy_lat
amazon_sample1_flashes.coords["Longitudes"] = dummy_lon
amazon_sample2_flashes.coords["Latitudes"] = dummy_lat
amazon_sample2_flashes.coords["Longitudes"] = dummy_lon
amazon_sample3_flashes.coords["Latitudes"] = dummy_lat
amazon_sample3_flashes.coords["Longitudes"] = dummy_lon

usa_LWI_train.coords["Latitudes"] = dummy_lat
usa_LWI_train.coords["Longitudes"] = dummy_lon
usa_sample1_LWI.coords["Latitudes"] = dummy_lat
usa_sample1_LWI.coords["Longitudes"] = dummy_lon
usa_sample2_LWI.coords["Latitudes"] = dummy_lat
usa_sample2_LWI.coords["Longitudes"] = dummy_lon
usa_sample3_LWI.coords["Latitudes"] = dummy_lat
usa_sample3_LWI.coords["Longitudes"] = dummy_lon

amazon_LWI_train.coords["Latitudes"] = dummy_lat
amazon_LWI_train.coords["Longitudes"] = dummy_lon
amazon_sample1_LWI.coords["Latitudes"] = dummy_lat
amazon_sample1_LWI.coords["Longitudes"] = dummy_lon
amazon_sample2_LWI.coords["Latitudes"] = dummy_lat
amazon_sample2_LWI.coords["Longitudes"] = dummy_lon
amazon_sample3_LWI.coords["Latitudes"] = dummy_lat
amazon_sample3_LWI.coords["Longitudes"] = dummy_lon

dec_us_feature.coords["Latitudes"] = dummy_lat
dec_us_feature.coords["Longitudes"] = dummy_lon
dec_amazon_feature.coords["Latitudes"] = dummy_lat
dec_amazon_feature.coords["Longitudes"] = dummy_lon
dec_us_flash.coords["Latitudes"] = dummy_lat
dec_us_flash.coords["Longitudes"] = dummy_lon
dec_amazon_flash.coords["Latitudes"] = dummy_lat
dec_amazon_flash.coords["Longitudes"] = dummy_lon
dec_us_LWI.coords["Latitudes"] = dummy_lat
dec_us_LWI.coords["Longitudes"] = dummy_lon
dec_amazon_LWI.coords["Latitudes"] = dummy_lat
dec_amazon_LWI.coords["Longitudes"] = dummy_lon

#Lopez:
usa_Lopez_train.coords["Latitudes"] = dummy_lat
usa_Lopez_train.coords["Longitudes"] = dummy_lon
amazon_Lopez_train.coords["Latitudes"] = dummy_lat
amazon_Lopez_train.coords["Longitudes"] = dummy_lon
usa_sample1_Lopez.coords["Latitudes"] = dummy_lat
usa_sample1_Lopez.coords["Longitudes"] = dummy_lon
amazon_sample1_Lopez.coords["Latitudes"] = dummy_lat
amazon_sample1_Lopez.coords["Longitudes"] = dummy_lon
usa_sample2_Lopez.coords["Latitudes"] = dummy_lat
usa_sample2_Lopez.coords["Longitudes"] = dummy_lon
amazon_sample2_Lopez.coords["Latitudes"] = dummy_lat
amazon_sample2_Lopez.coords["Longitudes"] = dummy_lon
usa_sample3_Lopez.coords["Latitudes"] = dummy_lat
usa_sample3_Lopez.coords["Longitudes"] = dummy_lon
amazon_sample3_Lopez.coords["Latitudes"] = dummy_lat
amazon_sample3_Lopez.coords["Longitudes"] = dummy_lon
dec_us_Lopez.coords["Latitudes"] = dummy_lat
dec_us_Lopez.coords["Longitudes"] = dummy_lon
dec_amazon_Lopez.coords["Latitudes"] = dummy_lat
dec_amazon_Lopez.coords["Longitudes"] = dummy_lon


In [39]:
#Okay, this is for the training. I am combining the amazon and the US data together. This is occuring BEFORE standardization. 
combined_train_features_us = xr.concat([us_features_train, usa_sample2_features, dec_us_feature], dim="Datetime") #This is an Xarray Dataset
combined_train_features_amazon = xr.concat([amazon_features_train, amazon_sample2_features, dec_amazon_feature], dim="Datetime") #This is an Xarray Dataset
combined_train_flashes_us = xr.concat([usa_flash_train, usa_sample2_flashes, dec_us_flash], dim="Datetime")
combined_train_flashes_amazon = xr.concat([amazon_flash_train, amazon_sample2_flashes, dec_amazon_flash], dim="Datetime")
combined_train_LWI_us = xr.concat([usa_LWI_train, usa_sample2_LWI, dec_us_LWI], dim="Datetime")
combined_train_LWI_amazon = xr.concat([amazon_LWI_train, amazon_sample2_LWI, dec_amazon_LWI], dim="Datetime")


In [40]:
#Total Training: 
total_training_features = xr.concat([combined_train_features_us, combined_train_features_amazon], dim="Datetime")
total_training_flashes = xr.concat([combined_train_flashes_us, combined_train_flashes_amazon], dim="Datetime")
total_training_LWI = xr.concat([combined_train_LWI_us, combined_train_LWI_amazon], dim="Datetime")

In [41]:
#Save out training
total_training_features.to_netcdf("total_training_features.nc")
total_training_flashes.to_netcdf("total_training_flashes.nc")
total_training_LWI.to_netcdf("total_training_LWI.nc")

In [42]:
#Now, do total testing: 
testing_features_us = xr.concat([usa_sample1_features, usa_sample3_features], dim="Datetime")
testing_features_amazon = xr.concat([amazon_sample1_features, amazon_sample3_features], dim="Datetime")
testing_flashes_us = xr.concat([usa_sample1_flashes, usa_sample3_flashes], dim="Datetime")
testing_flashes_amazon = xr.concat([amazon_sample1_flashes, amazon_sample3_flashes], dim="Datetime")
testing_LWI_us = xr.concat([usa_sample1_LWI, usa_sample3_LWI], dim="Datetime")
testing_LWI_amazon = xr.concat([amazon_sample1_LWI, amazon_sample3_LWI], dim="Datetime")
testing_Lopez_us = xr.concat([usa_sample1_Lopez, usa_sample3_Lopez],  dim="Datetime")
testing_Lopez_amazon = xr.concat([amazon_sample1_Lopez, amazon_sample3_Lopez],  dim="Datetime")

In [43]:
#Save out testing
testing_features_us.to_netcdf("testing_features_us.nc")
testing_features_amazon.to_netcdf("testing_features_amazon.nc")
testing_flashes_us.to_netcdf("testing_flashes_us.nc")
testing_flashes_amazon.to_netcdf("testing_flashes_amazon.nc")
testing_LWI_us.to_netcdf("testing_LWI_us.nc")
testing_LWI_amazon.to_netcdf("testing_LWI_amazon.nc")
testing_Lopez_us.to_netcdf("testing_Lopez_us.nc")
testing_Lopez_amazon.to_netcdf("testing_Lopez_amazon.nc")
